In [6]:
# =========================
# 0) Setup e imports - Competencia Data Mining UBA 2025
# =========================
!pip -q install polars==1.7.1 sympy matplotlib seaborn google-cloud-storage

import polars as pl
from pathlib import Path
from types import SimpleNamespace
import json, numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
from sympy import primerange
from google.cloud import storage

# Seeds oficiales de la competencia
SEEDS = [509963, 739373, 794341, 900623, 917827]
# Semillas adicionales (números primos grandes)
SEEDS_EXTRA = [1299827, 1500007, 1700021, 1900009, 2100013]
SEEDS_ALL = SEEDS + SEEDS_EXTRA  # 10 semillas totales

# Paths definitivos en Google Cloud / repo git
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
CSV_PATH    = PROJECT_ROOT / "datasets" / "competencia_02_crudo.csv"
SCHEMA_PATH = NOTEBOOK_DIR / "schema.json"
BASE_DIR    = NOTEBOOK_DIR / "dmeyf2025"

BUCKET_NAME = "nicocaviglia031_bukito3"
BUCKET_LOCAL_DIR = PROJECT_ROOT / BUCKET_NAME
BUCKET_LOCAL_DIR.mkdir(parents=True, exist_ok=True)


In [7]:
# =========================
# 1) Lectura de DF con Schema
# =========================

# Cargar schema
with open(SCHEMA_PATH) as f:
    schema_dict = json.load(f)

# Convertir strings a dtypes de Polars
schema = {k: getattr(pl, v) for k, v in schema_dict.items()}

# Lectura estricta (sin inferir tipos)
df = pl.read_csv(CSV_PATH, schema_overrides=schema)
df = df.rechunk()  # Compactar memoria

drop_cols = ["mprestamos_personales", "cprestamos_personales"]
df = df.drop(drop_cols)

df.shape

In [8]:
# =========================
# 1.1) Canaritos + Setup de zLGBM
# =========================

def create_canaritos(df: pl.DataFrame, qcanaritos: int = 5) -> pl.DataFrame:
    """
    Añade un número específico de columnas "canarito" (features aleatorias)
    a un DataFrame de Polars.

    Estas nuevas columnas contendrán valores aleatorios uniformes (entre 0 y 1)
    y se colocarán al principio del DataFrame, manteniendo el orden
    original de las demás columnas.

    Args:
        df (pl.DataFrame): El DataFrame de Polars al que se le añadirán
                           las columnas.
        qcanaritos (int): El número de columnas "canarito" que se
                            desea crear (ej: 5).

    Returns:
        pl.DataFrame: Un nuevo DataFrame con las columnas "canarito" añadidas
                      al principio.
    """

    # 1. Guardar los nombres de las columnas originales
    original_cols = df.columns
    num_filas = df.height
    # 2. Generar la lista de nombres para las nuevas columnas "canarito"
    canary_cols = [f"canarito_{i}" for i in range(1, qcanaritos + 1)]

    # 3. Crear las expresiones Polars para generar los números aleatorios
    #    pl.rand_uniform(0, 1) es el equivalente a runif()
    canary_expressions = [pl.lit(np.random.rand(num_filas)).alias(name) for name in canary_cols]

    # 4. Añadir las nuevas columnas y reordenar todo en un solo paso
    #    Usamos .select() para el reordenamiento final
    df = df.with_columns(
        canary_expressions
    ).select(
        canary_cols + original_cols  # Concatena listas para el nuevo orden
    )

    return df

shape: (32, 4)
┌──────────┬──────────┬────────┬────────┐
│ foto_mes ┆ CONTINUA ┆ BAJA+2 ┆ BAJA+1 │
│ ---      ┆ ---      ┆ ---    ┆ ---    │
│ i64      ┆ u32      ┆ u32    ┆ u32    │
╞══════════╪══════════╪════════╪════════╡
│ 201901   ┆ 122918   ┆ 720    ┆ 635    │
│ 201902   ┆ 123985   ┆ 693    ┆ 723    │
│ 201903   ┆ 124535   ┆ 738    ┆ 694    │
│ 201904   ┆ 125293   ┆ 502    ┆ 743    │
│ 201905   ┆ 126016   ┆ 681    ┆ 505    │
│ …        ┆ …        ┆ …      ┆ …      │
│ 202104   ┆ 161340   ┆ 1126   ┆ 952    │
│ 202105   ┆ 161946   ┆ 842    ┆ 1129   │
│ 202106   ┆ 162336   ┆ 1135   ┆ 842    │
│ 202107   ┆ 163459   ┆ 0      ┆ 1137   │
│ 202108   ┆ 164822   ┆ 0      ┆ 0      │
└──────────┴──────────┴────────┴────────┘


In [9]:
# =========================
# 1.2) Creacion de Canaritos
# =========================

plocal = SimpleNamespace(
    qcanaritos=5,
    min_data_in_leaf=20,
    learning_rate=1.0,
    gradient_bound=0.1,
    APO=5,
    ksemillerio=1,
)

df = create_canaritos(df, plocal.qcanaritos)

In [ ]:
# =========================
# 2) Construcción de clase_ternaria (baseline del profesor)
# =========================

def agregar_clase_ternaria_optimizada(df: pl.DataFrame) -> pl.DataFrame:
    """
    Versión optimizada usando Polars - Baseline del profesor
    """
    return (
        df.lazy()
        .sort(["numero_de_cliente", "foto_mes"])
        .with_columns([
            pl.when(
                pl.col("foto_mes").shift(-1).over("numero_de_cliente").is_null() &
                (pl.col("foto_mes") != pl.col("foto_mes").max())
            ).then(pl.lit("BAJA+1"))
            .when(
                pl.col("foto_mes").shift(-2).over("numero_de_cliente").is_null() &
                (pl.col("foto_mes") <= pl.col("foto_mes").max() - 2)
            ).then(pl.lit("BAJA+2"))
            .otherwise(pl.lit("CONTINUA"))
            .alias("clase_ternaria")
        ])
        .collect()
    )

# Agregar la columna clase_ternaria
df = agregar_clase_ternaria_optimizada(df)

# Tabla de frecuencia de clase_ternaria por foto_mes
tabla_clases = (
    df.pivot(
        values="numero_de_cliente",
        index="foto_mes",
        on="clase_ternaria",
        aggregate_function="len"
    )
    .fill_null(0)
)

print(tabla_clases)

In [ ]:
# =========================
# Feature engineering: agregados de montos/contadores
# =========================

# Totales de saldos en pesos
SALDOS_PESOS = [
    "mcuenta_corriente",
    "mcaja_ahorro",
    "mcuenta_corriente_adicional",
    "mcaja_ahorro_adicional",
    "mcuentas_saldo",
]

# Totales de saldos en dólares
SALDOS_DOLARES = [
    "mcaja_ahorro_dolares",
]

# Consumo tarjetas pesos
CONSUMO_TARJETAS_PESOS = [
    "mtarjeta_visa_consumo",
    "mtarjeta_master_consumo",
]

# Consumo tarjetas dólares
CONSUMO_TARJETAS_DOLARES = [
    "Visa_mconsumosdolares",
    "Master_mconsumosdolares",
]

# Pagos recurrentes pesos
PAGOS_RECURRENTE_PESOS = [
    "mpagodeservicios",
    "mpagomiscuentas",
    "mcuenta_debitos_automaticos",
    "mttarjeta_visa_debitos_automaticos",
    "mttarjeta_master_debitos_automaticos",
]

# Sumas de productos de inversión y plazos fijos pesos
INVERSIONES_PESOS = [
    "mplazo_fijo_pesos",
    "minversion1_pesos",
    "minversion2",
]

# Sumas de productos de inversión y plazos fijos dólares
INVERSIONES_DOLARES = [
    "mplazo_fijo_dolares",
    "minversion1_dolares",
]

# Endeudamiento total pesos
ENDEUDAMIENTO_PESOS = [
    "mprestamos_prendarios",
    "mprestamos_hipotecarios",
    "Master_msaldopesos",
    "Visa_msaldopesos",
    "Master_madelantopesos",
    "Visa_madelantopesos",
]

# Endeudamiento total dólares
ENDEUDAMIENTO_DOLARES = [
    "Master_msaldodolares",
    "Visa_msaldodolares",
    "Master_madelantodolares",
    "Visa_madelantodolares",
]

# Ingresos payroll pesos
PAYROLL_PESOS = [
    "mpayroll",
    "mpayroll2",
]

# Límites de compra
LIMITES_COMPRA = [
    "Master_mlimitecompra",
    "Visa_mlimitecompra",
]

# Suma de seguros contratados
SEGUROS_COUNT = [
    "cseguro_vida",
    "cseguro_auto",
    "cseguro_vivienda",
    "cseguro_accidentes_personales",
]

# Totales de transacciones (conteo)
TRANSACCIONES_COUNT = [
    "ctarjeta_debito_transacciones",
    "ctarjeta_visa_transacciones",
    "ctarjeta_master_transacciones",
    "chomebanking_transacciones",
    "cmobile_app_trx",
    "ccallcenter_transacciones",
    "catm_trx",
    "catm_trx_other",
    "ccajas_transacciones",
]

def sum_columns_safe(df: pl.DataFrame, cols: list[str]) -> pl.Series:
    existentes = [c for c in cols if c in df.columns]
    if not existentes:
        return pl.lit(0)
    return pl.sum_horizontal([pl.col(c) for c in existentes])

fe_columns = {
    "m_saldo_total_pesos": SALDOS_PESOS,
    "m_saldo_total_dolares": SALDOS_DOLARES,
    "m_consumo_tarjetas_pesos": CONSUMO_TARJETAS_PESOS,
    "m_consumo_tarjetas_dolares": CONSUMO_TARJETAS_DOLARES,
    "m_pagos_recurrentes_pesos": PAGOS_RECURRENTE_PESOS,
    "m_inversiones_pesos": INVERSIONES_PESOS,
    "m_inversiones_dolares": INVERSIONES_DOLARES,
    "m_endeudamiento_pesos": ENDEUDAMIENTO_PESOS,
    "m_endeudamiento_dolares": ENDEUDAMIENTO_DOLARES,
    "m_payroll_total": PAYROLL_PESOS,
    "m_limite_compra_total": LIMITES_COMPRA,
    "c_seguros_total": SEGUROS_COUNT,
    "c_transacciones_total": TRANSACCIONES_COUNT,
}

df = df.with_columns([
    sum_columns_safe(df, cols).alias(col_name)
    for col_name, cols in fe_columns.items()
])

df.shape

In [ ]:
# =========================
# 3) Feature Engineering Histórico - Lags y DeltaLags
# =========================

# Excluir IDs y metadata
EXCLUDE = {"numero_de_cliente", "foto_mes", "clase_ternaria"}

# Identificar columnas numéricas
num_cols = [c for c, dt in zip(df.columns, df.dtypes)
            if c not in EXCLUDE and dt.is_numeric()]

def add_lags_and_deltas(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
    """
    Agregar lags y deltas de orden 1 y 2 - Baseline del profesor
    """
    out = df.sort(["numero_de_cliente", "foto_mes"]).lazy()

    # Lags 1 y 2
    out = out.with_columns([
        pl.col(c).shift(1).over("numero_de_cliente").alias(f"{c}_lag1") for c in cols
    ] + [
        pl.col(c).shift(2).over("numero_de_cliente").alias(f"{c}_lag2") for c in cols
    ])

    # Delta 1: x_t - x_{t-1} ; Delta 2: x_t - x_{t-2}
    out = out.with_columns([
        (pl.col(c) - pl.col(f"{c}_lag1")).alias(f"{c}_d1") for c in cols
    ] + [
        (pl.col(c) - pl.col(f"{c}_lag2")).alias(f"{c}_d2") for c in cols
    ])

    return out.collect()

# Aplicar lags y deltas
df = add_lags_and_deltas(df, num_cols)

df.shape

In [ ]:
# Configuración de la competencia
TRAIN_MESES = [201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104]  # Meses de entrenamiento

TEST_MES   = 202106                            # Mes de aplicación

APPLY_MES = 202108                             # Mes de test

In [ ]:
# =========================
# 4) Undersampling de clientes CONTINUA repetidos en todos los meses de entrenamiento
# =========================

TRAIN_MESES_SET = set(TRAIN_MESES)
TOTAL_MESES_TRAIN = len(TRAIN_MESES_SET)

train_subset = df.filter(pl.col("foto_mes").is_in(TRAIN_MESES))

clientes_continua_full = (
    train_subset
    .group_by("numero_de_cliente")
    .agg([
        pl.col("clase_ternaria").eq("CONTINUA").all().alias("solo_continua"),
        pl.col("foto_mes").n_unique().alias("n_meses"),
    ])
    .filter(
        pl.col("solo_continua") &
        (pl.col("n_meses") == TOTAL_MESES_TRAIN)
    )
    .select("numero_de_cliente")
)

clientes_continua_list = clientes_continua_full.get_column("numero_de_cliente").to_list()

fraccion_sample = 0.04
n_total_continua = len(clientes_continua_list)
if n_total_continua > 0:
    n_sample = max(1, int(np.ceil(n_total_continua * fraccion_sample)))
    clientes_sample = clientes_continua_full.sample(n=n_sample, seed=SEEDS[0])
    clientes_sample_list = clientes_sample.get_column("numero_de_cliente").to_list()
else:
    clientes_sample_list = []

clientes_a_excluir = set(clientes_continua_list) - set(clientes_sample_list)

print(f"Clientes CONTINUA en todos los meses de train: {n_total_continua}")
print(f"Clientes CONTINUA retenidos (4%): {len(clientes_sample_list)}")

_df_train = train_subset.filter(~pl.col("numero_de_cliente").is_in(list(clientes_a_excluir)))
_df_test = df.filter(pl.col("foto_mes") == TEST_MES)
_df_apply = df.filter(pl.col("foto_mes") == APPLY_MES)


In [ ]:
print("_df_train shape:", _df_train.shape)
print("_df_test shape:", _df_test.shape)
print("_df_apply shape:", _df_apply.shape)
print("Primeras columnas en df      :", df.columns[:10])
print("Primeras columnas en _df_train:", _df_train.columns[:10])

In [ ]:
# =========================
# 5) Entrenamiento LightGBM con métrica de ganancia
# =========================

plocal = SimpleNamespace(
    qcanaritos=5,
    min_data_in_leaf=20,
    learning_rate=1.0,
    gradient_bound=0.1,
    APO=5,
    ksemillerio=1,
)

PARAM = SimpleNamespace(
    semilla_primigenia=SEEDS[0],
    qcanaritos=plocal.qcanaritos,
)

# Constantes de ganancia
ganancia_acierto = 780000
costo_estimulo = 20000

# Función de evaluación personalizada

def lgb_gan_eval(y_pred, data):
    weight = data.get_weight()
    if weight is None:
        weight = np.ones_like(y_pred)
    ganancia = (
        np.where(weight == 1.00002, ganancia_acierto, 0)
        - np.where(weight < 1.00002, costo_estimulo, 0)
    )
    ganancia = ganancia[np.argsort(y_pred)[::-1]]
    ganancia = np.cumsum(ganancia)
    return "gan_eval", float(np.max(ganancia)), True


train_pd = _df_train.to_pandas()
test_pd = _df_test.to_pandas()
apply_pd = _df_apply.to_pandas()

rng = np.random.default_rng(PARAM.semilla_primigenia)

drop_cols = {"numero_de_cliente", "foto_mes", "clase_ternaria"}
feature_cols = [c for c in train_pd.columns if c not in drop_cols]

print("=== Dataset shapes ===")
print(f"train: {train_pd.shape} | test: {test_pd.shape} | apply: {apply_pd.shape}")
print(f"Features disponibles: {len(feature_cols)} (incluye canaritos={plocal.qcanaritos})")

y_train = train_pd["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)
y_valid = test_pd["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)

weight_train = np.where(y_train == 1, 1.00002, 1.0)
weight_valid = np.where(y_valid == 1, 1.00002, 1.0)

lgb_train = lgb.Dataset(train_pd[feature_cols], label=y_train, weight=weight_train)

lgbm_param_completo = {
    "boosting": "gbdt",
    "objective": "binary",
    "metric": "None",
    "first_metric_only": False,
    "boost_from_average": True,
    "feature_pre_filter": False,
    "force_row_wise": True,
    "verbosity": -100,
    "seed": PARAM.semilla_primigenia,
    "max_bin": 31,
    "min_data_in_leaf": plocal.min_data_in_leaf,
    "num_leaves": 9999,
    "learning_rate": plocal.learning_rate,
    "feature_fraction": 0.50,
    "canaritos": PARAM.qcanaritos,
    "gradient_bound": plocal.gradient_bound,
}

num_boost_round = 9999

print("\n=== Hyperparámetros principales ===")
print(
    f"seed={lgbm_param_completo['seed']} | min_leaf={plocal.min_data_in_leaf} | "
    f"learning_rate={plocal.learning_rate} | num_leaves={lgbm_param_completo['num_leaves']}"
)
print(
    f"canaritos={PARAM.qcanaritos} | gradient_bound={plocal.gradient_bound} | "
    f"feature_fraction={lgbm_param_completo['feature_fraction']} | rounds={num_boost_round}"
)

model = lgb.train(
    lgbm_param_completo,
    train_set=lgb_train,
    num_boost_round=num_boost_round,
)

best_iteration = model.best_iteration or num_boost_round
apply_pd["pred_lgbm"] = model.predict(apply_pd[feature_cols], num_iteration=best_iteration)

best_gain = model.best_score.get("valid", {}).get("gan_eval")
print(f"Best iteration: {best_iteration}")
if best_gain is not None:
    print(f"Mejor ganancia (valid): {best_gain:.0f}")

valid_scores = model.predict(test_pd[feature_cols], num_iteration=best_iteration)


In [ ]:
# =========================
# 6) Predicciones binarias finales y archivo submit
# =========================

models = [model]
predictions_per_seed = [apply_pd["pred_lgbm"].to_numpy()]

predictions_matrix = np.vstack(predictions_per_seed)
final_predictions = np.mean(predictions_matrix, axis=0)

clientes_ordenados = np.argsort(final_predictions)[::-1]

n_clientes_objetivo = 11000
predicciones_binarias = np.zeros(len(final_predictions), dtype=int)
predicciones_binarias[clientes_ordenados[:n_clientes_objetivo]] = 1

id_apply = apply_pd["numero_de_cliente"].to_numpy()
submit_df = pl.DataFrame(
    {
        "numero_de_cliente": id_apply[clientes_ordenados],
        "Predicted": predicciones_binarias[clientes_ordenados],
    }
)

submit_path = BASE_DIR / "submit_lgbm_apply.csv"
submit_path.parent.mkdir(parents=True, exist_ok=True)
submit_df.write_csv(submit_path)

print("=== Submit APPLY listo ===")
print(f"Archivo: {submit_path}")
print(f"Clientes positivos: {predicciones_binarias.sum()} de {len(predicciones_binarias)}")

clientes_valid_ordenados = np.argsort(valid_scores)[::-1]
predicciones_binarias_valid = np.zeros(len(valid_scores), dtype=int)
predicciones_binarias_valid[clientes_valid_ordenados[:n_clientes_objetivo]] = 1

id_valid = test_pd["numero_de_cliente"].to_numpy()
submit_valid_df = pl.DataFrame(
    {
        "numero_de_cliente": id_valid[clientes_valid_ordenados],
        "Predicted": predicciones_binarias_valid[clientes_valid_ordenados],
    }
)

submit_valid_path = BASE_DIR / "submit_lgbm_test202106.csv"
submit_valid_df.write_csv(submit_valid_path)

print("\n=== Submit TEST 202106 listo ===")
print(f"Archivo: {submit_valid_path}")
print(f"Clientes positivos: {predicciones_binarias_valid.sum()} de {len(predicciones_binarias_valid)}")